In [1]:
from __future__ import annotations

import json
import os
import signal
import subprocess
import time
from pathlib import Path

import httpx
from IPython.display import HTML, display


SOLUTION_ROOT = Path(
    "/home/jovyan/chest-xray-ai-assistant"
)

DATA_ROOT = Path(
    "/home/jovyan/apicdsa2-datavol-1/"
    "chest-xray-ai-assistant-data"
)

API_OUTPUT_ROOT = (
    DATA_ROOT / "outputs" / "api"
)

FASTAPI_LOG_PATH = (
    API_OUTPUT_ROOT
    / "swagger_demo_fastapi_backend.log"
)

FASTAPI_HOST = "127.0.0.1"
FASTAPI_PORT = 8000

FASTAPI_BASE_URL = (
    f"http://{FASTAPI_HOST}:{FASTAPI_PORT}"
)

NB_PREFIX = os.environ.get(
    "NB_PREFIX",
    "",
).rstrip("/")

if not NB_PREFIX:
    raise RuntimeError(
        "NB_PREFIX is unavailable. The Kubeflow notebook "
        "proxy path cannot be constructed."
    )

FASTAPI_PROXY_ROOT = (
    f"{NB_PREFIX}/proxy/{FASTAPI_PORT}"
)

SWAGGER_PROXY_PATH = (
    f"{FASTAPI_PROXY_ROOT}/docs"
)

REDOC_PROXY_PATH = (
    f"{FASTAPI_PROXY_ROOT}/redoc"
)

OPENAPI_PROXY_PATH = (
    f"{FASTAPI_PROXY_ROOT}/openapi.json"
)


def fastapi_is_healthy() -> bool:
    try:
        response = httpx.get(
            f"{FASTAPI_BASE_URL}/health",
            timeout=5.0,
        )

        return response.status_code == 200

    except httpx.RequestError:
        return False


def matching_fastapi_process_ids() -> list[int]:
    """Identify only the Uvicorn process serving this API."""

    matching_process_ids = []

    for process_root in Path("/proc").iterdir():
        if not process_root.name.isdigit():
            continue

        command_path = process_root / "cmdline"

        try:
            command_parts = command_path.read_bytes().split(
                b"\0"
            )

            command_text = " ".join(
                part.decode(
                    "utf-8",
                    errors="replace",
                )
                for part in command_parts
                if part
            )

        except (
            FileNotFoundError,
            PermissionError,
            ProcessLookupError,
        ):
            continue

        if (
            "uvicorn" in command_text
            and "api.main:app" in command_text
            and "--port" in command_text
            and str(FASTAPI_PORT) in command_text
        ):
            matching_process_ids.append(
                int(process_root.name)
            )

    return sorted(
        matching_process_ids
    )


# Confirm that the API's actual OpenAPI contract is valid
current_openapi_response = httpx.get(
    f"{FASTAPI_BASE_URL}/openapi.json",
    timeout=15.0,
)

current_openapi_response.raise_for_status()

current_openapi = (
    current_openapi_response.json()
)

OPENAPI_VERSION = current_openapi.get(
    "openapi"
)

OPENAPI_PATH_COUNT = len(
    current_openapi.get(
        "paths",
        {},
    )
)

if (
    not isinstance(OPENAPI_VERSION, str)
    or not OPENAPI_VERSION.startswith("3.")
):
    raise RuntimeError(
        "The local FastAPI OpenAPI document does not "
        "contain a valid OpenAPI 3.x version."
    )

if OPENAPI_PATH_COUNT != 12:
    raise RuntimeError(
        "The local FastAPI OpenAPI document does not "
        "contain the expected twelve paths."
    )


# Stop only the existing FastAPI Uvicorn process
existing_fastapi_process_ids = (
    matching_fastapi_process_ids()
)

if not existing_fastapi_process_ids:
    raise RuntimeError(
        "The running FastAPI Uvicorn process could not "
        "be identified safely. No process was stopped."
    )

for process_id in existing_fastapi_process_ids:
    os.kill(
        process_id,
        signal.SIGTERM,
    )

shutdown_deadline = time.monotonic() + 30

while (
    fastapi_is_healthy()
    and time.monotonic() < shutdown_deadline
):
    time.sleep(0.5)

if fastapi_is_healthy():
    raise RuntimeError(
        "The existing FastAPI process did not stop cleanly."
    )


# Restart FastAPI with the authenticated proxy root
fastapi_environment = os.environ.copy()

fastapi_environment.update(
    {
        "HF_HOME": str(
            DATA_ROOT / "hf-cache"
        ),
        "HF_DATASETS_CACHE": str(
            DATA_ROOT
            / "hf-cache"
            / "datasets"
        ),
        "TORCH_HOME": str(
            DATA_ROOT
            / "models"
            / "torch-cache"
        ),
        "MLFLOW_TRACKING_URI": (
            f"file://{DATA_ROOT / 'mlflow'}"
        ),
        "TOKENIZERS_PARALLELISM": "false",
        "PYTHONUNBUFFERED": "1",
        "PYTHONPATH": str(SOLUTION_ROOT),
    }
)

fastapi_log_handle = FASTAPI_LOG_PATH.open(
    "w",
    encoding="utf-8",
)

fastapi_process = subprocess.Popen(
    [
        "/opt/conda/bin/python",
        "-m",
        "uvicorn",
        "api.main:app",
        "--host",
        FASTAPI_HOST,
        "--port",
        str(FASTAPI_PORT),
        "--root-path",
        FASTAPI_PROXY_ROOT,
        "--proxy-headers",
        "--forwarded-allow-ips",
        "*",
    ],
    cwd=str(SOLUTION_ROOT),
    env=fastapi_environment,
    stdout=fastapi_log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

fastapi_log_handle.close()

startup_deadline = time.monotonic() + 180

while time.monotonic() < startup_deadline:
    if fastapi_process.poll() is not None:
        log_tail = FASTAPI_LOG_PATH.read_text(
            encoding="utf-8",
            errors="replace",
        )[-5000:]

        raise RuntimeError(
            "FastAPI exited during proxy-aware startup.\n\n"
            f"{log_tail}"
        )

    if fastapi_is_healthy():
        break

    time.sleep(1.0)

if not fastapi_is_healthy():
    raise RuntimeError(
        "FastAPI did not become healthy within "
        "180 seconds."
    )


# Validate the proxy-aware Swagger configuration
swagger_html_response = httpx.get(
    f"{FASTAPI_BASE_URL}/docs",
    timeout=15.0,
)

swagger_html_response.raise_for_status()

EXPECTED_PROXY_OPENAPI_URL = (
    f"{FASTAPI_PROXY_ROOT}/openapi.json"
)

SWAGGER_PROXY_CONFIGURATION_VALID = (
    EXPECTED_PROXY_OPENAPI_URL
    in swagger_html_response.text
)

if not SWAGGER_PROXY_CONFIGURATION_VALID:
    raise RuntimeError(
        "Swagger does not contain the expected "
        "proxy-aware OpenAPI URL."
    )

final_openapi_response = httpx.get(
    f"{FASTAPI_BASE_URL}/openapi.json",
    timeout=15.0,
)

final_openapi_response.raise_for_status()

final_openapi = final_openapi_response.json()

if final_openapi.get("openapi") != OPENAPI_VERSION:
    raise RuntimeError(
        "The OpenAPI version changed during the restart."
    )

if len(final_openapi.get("paths", {})) != 12:
    raise RuntimeError(
        "The twelve-path OpenAPI contract was not preserved."
    )


print("PROXY-AWARE FASTAPI DOCUMENTATION")
print("-" * 100)
print(f"FastAPI process ID       : {fastapi_process.pid}")
print(f"FastAPI health           : PASS")
print(f"OpenAPI version          : {OPENAPI_VERSION}")
print(f"OpenAPI paths            : {OPENAPI_PATH_COUNT} / 12")
print(f"Proxy root               : {FASTAPI_PROXY_ROOT}")
print(f"Swagger proxy URL        : {SWAGGER_PROXY_PATH}")
print(f"ReDoc proxy URL          : {REDOC_PROXY_PATH}")
print(f"OpenAPI JSON proxy URL   : {OPENAPI_PROXY_PATH}")
print(f"Swagger proxy routing    : PASS")
print(f"Streamlit restarted      : NO")
print(f"Models retrained         : NO")
print(f"Backend log              : {FASTAPI_LOG_PATH}")
print()
print("READY FOR SWAGGER API DEMONSTRATION")


display(
    HTML(
        f"""
        <div style="
            padding: 14px 16px;
            border: 1px solid #d0d7de;
            border-radius: 8px;
            background: #f6f8fa;
            margin-top: 12px;
        ">
            <strong>API documentation:</strong><br><br>

            <a
                href="{SWAGGER_PROXY_PATH}"
                target="_blank"
                style="margin-right: 20px;"
            >
                Open Swagger UI
            </a>

            <a
                href="{REDOC_PROXY_PATH}"
                target="_blank"
                style="margin-right: 20px;"
            >
                Open ReDoc
            </a>

            <a
                href="{OPENAPI_PROXY_PATH}"
                target="_blank"
            >
                Open OpenAPI JSON
            </a>
        </div>
        """
    )
)

PROXY-AWARE FASTAPI DOCUMENTATION
----------------------------------------------------------------------------------------------------
FastAPI process ID       : 1724
FastAPI health           : PASS
OpenAPI version          : 3.1.0
OpenAPI paths            : 12 / 12
Proxy root               : /notebook/2024ac05653-s2-25-aimlczg536/apicdsa2/proxy/8000
Swagger proxy URL        : /notebook/2024ac05653-s2-25-aimlczg536/apicdsa2/proxy/8000/docs
ReDoc proxy URL          : /notebook/2024ac05653-s2-25-aimlczg536/apicdsa2/proxy/8000/redoc
OpenAPI JSON proxy URL   : /notebook/2024ac05653-s2-25-aimlczg536/apicdsa2/proxy/8000/openapi.json
Swagger proxy routing    : PASS
Streamlit restarted      : NO
Models retrained         : NO
Backend log              : /home/jovyan/apicdsa2-datavol-1/chest-xray-ai-assistant-data/outputs/api/swagger_demo_fastapi_backend.log

READY FOR SWAGGER API DEMONSTRATION
